# NeuroVolume: what does MRI add?

An executed walkthrough of the saved OASIS-2 experiment. This notebook reads measured artifacts and runs synthetic inference; it does not silently retrain or change the held-out split. See `docs/PROTOCOL.md` for the pre-result protocol.

In [1]:
import json
from pathlib import Path
import pandas as pd
root = Path.cwd()
report = json.loads((root / 'reports/results.json').read_text())
print(json.dumps(report['cohort'], indent=2))

{
  "participants": 150,
  "visits": 373,
  "development": 120,
  "test": 30,
  "baseline_positive": 65,
  "baseline_negative": 85,
  "test_positive": 13,
  "test_negative": 17,
  "baseline_age_range": [
    60,
    96
  ]
}


## Scientific question

Does adding MRI-derived intracranial and normalized whole-brain volume improve first-visit CDR > 0 classification beyond demographics? We also test MRI above demographics + MMSE. The 373 original sessions are not 373 independent people. CDR > 0 is an operational impairment label, not confirmed Alzheimer pathology or future decline.

In [2]:
from neurovolume.core import FEATURES
for name, features in FEATURES.items():
    print(name, ':', ', '.join(features))

demographics : Age, Sex, EDUC, SES
mri : Age, Sex, EDUC, SES, eTIV, nWBV
cognition : Age, Sex, EDUC, SES, MMSE
combined : Age, Sex, EDUC, SES, eTIV, nWBV, MMSE


## All predefined models

Logistic regression is the primary comparison. Tree models are benchmarks; no winner is selected using test accuracy. Development CV: 5 folds × 3 repeats. The test set has 30 participants.

In [3]:
rows = []
for item in report['results']:
    rows.append({'model': item['key'], 'development_auc': item['cv_auc_mean'], 'test_auc': item['test']['auc'], 'test_accuracy': item['test']['accuracy']})
print(pd.DataFrame(rows).round(3).to_string(index=False))

                model  development_auc  test_auc  test_accuracy
demographics_logistic            0.677     0.683          0.667
  demographics_forest            0.679     0.661          0.567
demographics_boosting            0.605     0.719          0.600
         mri_logistic            0.744     0.688          0.667
           mri_forest            0.703     0.670          0.600
         mri_boosting            0.688     0.638          0.633
   cognition_logistic            0.882     0.919          0.800
     cognition_forest            0.894     0.910          0.867
   cognition_boosting            0.880     0.905          0.867
    combined_logistic            0.898     0.891          0.800
      combined_forest            0.886     0.914          0.900
    combined_boosting            0.872     0.919          0.900


## Incremental MRI value

The uncertainty intervals below condition on the fitted models and class balance. They are paired, participant bootstrap intervals, not evidence of transportability to a new hospital.

In [4]:
for d in report['differences']:
    print(f"{d['label']}: AUC delta {d['auc_delta']:+.3f}; 95% interval [{d['ci'][0]:.3f}, {d['ci'][1]:.3f}]")

MRI added to demographics: AUC delta +0.005; 95% interval [-0.158, 0.167]
MRI added to demographics + cognition: AUC delta -0.027; 95% interval [-0.113, 0.041]


![Model comparison](reports/model-comparison.png)

MRI improved development CV AUC but showed no clear held-out improvement. Cognitive testing improves classification of a related clinical construct; this is not evidence of detection before symptoms.

## Use an actual trained model

The following values are invented for demonstration. No real participant record is included. The output is a research score, not a calibrated clinical probability.

In [5]:
from neurovolume.core import predict_portable
models = json.loads((root / 'models/logistic.json').read_text())
example = json.loads((root / 'examples/synthetic.json').read_text())
for family, model in models.items():
    score = predict_portable(model, example)
    print(f'{family}: {score:.4f}')

demographics: 0.2395
mri: 0.2755
cognition: 0.4429
combined: 0.4645


## Next phase

Future decline is not trained here. Irregular follow-up requires an endpoint, a horizon, and censoring rules. A larger event count is preferable for a meaningful prognostic model.

In [6]:
print(json.dumps(report['future_feasibility'], indent=2))

{
  "initially_cdr_zero": 85,
  "later_cdr_positive": 15,
  "status": "Feasibility count only; no prognostic model trained."
}


## Reproduce or extend

Obtain authorized OASIS-2 data, then run `python -m neurovolume.train --data data/oasis2.xlsx`, `python scripts/sync_demo.py`, and `python scripts/make_figures.py`. Do not tune against this now-observed test set. Read the model card and data terms before reuse.

Data were provided by OASIS. OASIS-2: Longitudinal; D. Marcus, R. Buckner, J. Csernansky, J. Morris; P50 AG05681, P01 AG03991, P01 AG026276, R01 AG021910, P20 MH071616, U24 RR021382. Source: https://doi.org/10.1162/jocn.2009.21407